# 04 RP3beta Variant Walkthrough

This notebook documents the RP3beta variant and its metadata-aware extensions. It is organized as a progression from the original interaction-only graph model to increasingly richer metadata integration.

## Variant progression
1. **Base RP3beta**: graph-based random-walk recommender over the train user-item matrix, implemented in `phase2/run_rp3beta.py`.
2. **Local hyperparameter check**: focused validation sweep around the retained `alpha`, `beta`, and item-neighbor settings.
3. **Simple category-blend prototype**: metadata prototype using category-only side information.
4. **Metadata implementations**: metadata similarity reranking/fusion in `phase2/run_rp3beta_metadata.py` and the richer content-aware path in `phase2/run_rp3beta_metadata_best.py`.

The purpose of this notebook is to make the full design path visible: base graph signal first, simple metadata idea second, and the final metadata implementations last.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from scipy import sparse
from sklearn.preprocessing import normalize

from analysis.shared_utils import PROCESSED_DIR, load_retained_results
from phase1.common import build_train_matrix, load_split_data, top_k_from_scores

ALPHA = 0.9
BETA = 0.4
SIMILARITY_TOPK = 400
BLOCK_SIZE = 256
TOP_K = 10
RUN_MODEL = False


def fit_rp3beta_local(train_matrix: sparse.csr_matrix, alpha: float, beta: float, similarity_topk: int, block_size: int) -> sparse.csr_matrix:
    p_ui = normalize(train_matrix, norm="l1", axis=1, copy=True)
    p_iu = normalize(train_matrix.T.tocsr(), norm="l1", axis=1, copy=True)

    if alpha != 1.0:
        p_ui.data = np.power(p_ui.data, alpha)
        p_iu.data = np.power(p_iu.data, alpha)

    item_degree = np.ravel(train_matrix.sum(axis=0)).astype(np.float64)
    degree_penalty = np.power(item_degree + 1e-12, -beta)
    degree_penalty[~np.isfinite(degree_penalty)] = 0.0

    num_items = train_matrix.shape[1]
    rows, cols, data = [], [], []
    for start in range(0, num_items, block_size):
        end = min(start + block_size, num_items)
        block_scores = (p_iu[start:end] @ p_ui).toarray().astype(np.float32, copy=False)
        block_scores *= degree_penalty[np.newaxis, :]
        for local_row in range(end - start):
            item_id = start + local_row
            row_scores = block_scores[local_row]
            row_scores[item_id] = 0.0
            candidate_idx = np.flatnonzero(row_scores > 0.0)
            if candidate_idx.size == 0:
                continue
            if candidate_idx.size > similarity_topk:
                top_idx = np.argpartition(row_scores[candidate_idx], -similarity_topk)[-similarity_topk:]
                candidate_idx = candidate_idx[top_idx]
            candidate_scores = row_scores[candidate_idx]
            order = np.argsort(candidate_scores)[::-1]
            rows.extend([item_id] * len(candidate_idx))
            cols.extend(candidate_idx[order].tolist())
            data.extend(candidate_scores[order].tolist())
    return sparse.csr_matrix((np.asarray(data, dtype=np.float32), (np.asarray(rows), np.asarray(cols))), shape=(num_items, num_items), dtype=np.float32)

retained_test = load_retained_results("test")
rp3_row = retained_test.loc[retained_test["model"] == "RP3beta-a0.9-b0.4-t400"]
display(Markdown("## Saved RP3beta test result"))
display(rp3_row)

if RUN_MODEL:
    split_data = load_split_data(PROCESSED_DIR, "test")
    train_matrix = build_train_matrix(split_data.train_df, split_data.num_users, split_data.num_items)
    similarity = fit_rp3beta_local(train_matrix, ALPHA, BETA, SIMILARITY_TOPK, BLOCK_SIZE)
    sample_user = split_data.eligible_users[0]
    scores = train_matrix[sample_user].dot(similarity).toarray().ravel()
    recs = top_k_from_scores(scores, split_data.train_user_items.get(sample_user, set()), TOP_K)
    print("Sample user:", sample_user)
    print("Top recommendations:", recs)
else:
    print("Set RUN_MODEL = True if you want to recompute RP3beta safely inside the notebook.")

## Saved RP3beta test result

,model,precision@10,recall@10,ndcg@10,users_evaluated,eval_split
4,RP3beta-a0.9-b0.4-t400,0.043121,0.064997,0.085032,7080,test


Set RUN_MODEL = True if you want to recompute RP3beta safely inside the notebook.


In [3]:
retained_test = retained_test.copy()
retained_test["delta_vs_UserCF_ndcg"] = retained_test["ndcg@10"] - retained_test.loc[retained_test["model"] == "UserCF", "ndcg@10"].iloc[0]
retained_test["delta_vs_UserCF_recall"] = retained_test["recall@10"] - retained_test.loc[retained_test["model"] == "UserCF", "recall@10"].iloc[0]
retained_test["delta_vs_UserCF_precision"] = retained_test["precision@10"] - retained_test.loc[retained_test["model"] == "UserCF", "precision@10"].iloc[0]

print("Base RP3beta versus the retained interaction-only field")
display(retained_test.sort_values("ndcg@10", ascending=False))


Base RP3beta versus the retained interaction-only field


,model,precision@10,recall@10,ndcg@10,users_evaluated,eval_split,delta_vs_UserCF_ndcg,delta_vs_UserCF_recall,delta_vs_UserCF_precision
3,EASE-Binary-3000,0.044746,0.065984,0.086413,7080,test,0.004452,0.002531,0.001992
4,RP3beta-a0.9-b0.4-t400,0.043121,0.064997,0.085032,7080,test,0.003072,0.001545,0.000367
5,ItemKNN-BM25-K320,0.042994,0.063505,0.083244,7080,test,0.001284,0.000053,0.000240
1,UserCF,0.042754,0.063452,0.081960,7080,test,0.000000,0.000000,0.000000
0,Popularity,0.037797,0.058550,0.071684,7080,test,-0.010276,-0.004902,-0.004958
2,SVD,0.022797,0.027715,0.038456,7080,test,-0.043504,-0.035737,-0.019958


In [4]:
from phase1.common import evaluate_topk

RUN_SWEEP = True

if RUN_SWEEP:
    split_data = load_split_data(PROCESSED_DIR, "val")
    train_matrix = build_train_matrix(split_data.train_df, split_data.num_users, split_data.num_items)
    param_grid = [
        {"alpha": 0.8, "beta": 0.4, "similarity_topk": 300},
        {"alpha": 0.9, "beta": 0.4, "similarity_topk": 400},
        {"alpha": 1.0, "beta": 0.4, "similarity_topk": 400},
        {"alpha": 0.9, "beta": 0.5, "similarity_topk": 400},
        {"alpha": 0.9, "beta": 0.4, "similarity_topk": 500},
    ]
    sweep_rows = []
    for params in param_grid:
        similarity = fit_rp3beta_local(
            train_matrix,
            alpha=params["alpha"],
            beta=params["beta"],
            similarity_topk=params["similarity_topk"],
            block_size=BLOCK_SIZE,
        )
        rec_cache = {}
        for user_id in split_data.eligible_users:
            scores = train_matrix[user_id].dot(similarity).toarray().ravel()
            rec_cache[user_id] = top_k_from_scores(scores, split_data.train_user_items.get(user_id, set()), TOP_K)
        metrics = evaluate_topk(
            lambda user_id, k: rec_cache[user_id][:k],
            split_data.eligible_users,
            split_data.eval_user_items,
            TOP_K,
            f"a={params['alpha']}, b={params['beta']}, t={params['similarity_topk']}"
        )
        sweep_rows.append(
            {
                "alpha": params["alpha"],
                "beta": params["beta"],
                "similarity_topk": params["similarity_topk"],
                "precision@10": metrics["precision_at_k"],
                "recall@10": metrics["recall_at_k"],
                "ndcg@10": metrics["ndcg_at_k"],
            }
        )
    display(pd.DataFrame(sweep_rows).sort_values("ndcg@10", ascending=False))
else:
    print("Set RUN_SWEEP = True only if you want a focused validation sweep around the retained RP3beta settings.")

,alpha,beta,similarity_topk,precision@10,recall@10,ndcg@10
4,0.9,0.4,500,0.039707,0.065172,0.081294
1,0.9,0.4,400,0.039826,0.065071,0.081088
2,1.0,0.4,400,0.039878,0.064876,0.080952
3,0.9,0.5,400,0.040275,0.064227,0.080517
0,0.8,0.4,300,0.039350,0.064217,0.080280


## Phase B: Simple Category-Blend Prototype

This section adds a lightweight metadata prototype directly inside the notebook. It keeps the base RP3beta graph score as the primary signal and blends it with a category-only metadata score derived from app categories in `app_info_sample.csv`.

The prototype is intentionally simple: it is not the final pushed repo implementation, but it documents the first natural hybrid step before moving to the richer metadata implementations. It also avoids the aggregate metadata fields that may introduce temporal leakage.

Progression captured here:

**base RP3beta -> simple category blend -> metadata similarity rerank/fusion -> richer low-rank content path**

In [5]:
from IPython.display import Markdown, display

import numpy as np
import pandas as pd

from analysis.shared_utils import PROCESSED_DIR
from phase1.common import build_train_matrix, evaluate_topk, load_split_data, top_k_from_scores
from phase2.metadata_utils import build_metadata_similarity
from phase2.run_rp3beta import fit_rp3beta as fit_rp3beta_base

RUN_SIMPLE_BLEND = True
ALPHA_LOCAL = 0.9
BETA_LOCAL = 0.4
SIMILARITY_TOPK_LOCAL = 400
BLOCK_SIZE_LOCAL = 256
TOP_K_LOCAL = 10
CATEGORY_SIM_TOPK = 200
CATEGORY_SIM_BATCH = 512
LAMBDA_VALUES = (0.6, 0.7, 0.8, 0.9)


def normalize_score_vector(scores: np.ndarray) -> np.ndarray:
    scores = np.asarray(scores, dtype=np.float64).copy()
    finite = np.isfinite(scores)
    if not finite.any():
        return scores
    vals = scores[finite]
    lo, hi = vals.min(), vals.max()
    if hi > lo:
        scores[finite] = (vals - lo) / (hi - lo)
    else:
        scores[finite] = 0.0
    return scores


if RUN_SIMPLE_BLEND:
    split_data = load_split_data(PROCESSED_DIR, "val")
    train_matrix = build_train_matrix(
        split_data.train_df,
        split_data.num_users,
        split_data.num_items,
    )

    rp3_similarity = fit_rp3beta_base(
        train_matrix,
        alpha=ALPHA_LOCAL,
        beta=BETA_LOCAL,
        similarity_topk=SIMILARITY_TOPK_LOCAL,
        block_size=BLOCK_SIZE_LOCAL,
    )

    category_similarity = build_metadata_similarity(
        split_data.num_items,
        mode="category",
        topk=CATEGORY_SIM_TOPK,
        batch_size=CATEGORY_SIM_BATCH,
    )

    blend_rows = []
    for lam in LAMBDA_VALUES:
        rec_cache = {}
        for user_id in split_data.eligible_users:
            base_scores = train_matrix[user_id].dot(rp3_similarity).toarray().ravel()
            category_scores = train_matrix[user_id].dot(category_similarity).toarray().ravel()

            base_scores = normalize_score_vector(base_scores)
            category_scores = normalize_score_vector(category_scores)
            combined_scores = lam * base_scores + (1.0 - lam) * category_scores

            rec_cache[user_id] = top_k_from_scores(
                combined_scores,
                split_data.train_user_items.get(user_id, set()),
                TOP_K_LOCAL,
            )

        metrics = evaluate_topk(
            lambda user_id, k: rec_cache[user_id][:k],
            split_data.eligible_users,
            split_data.eval_user_items,
            TOP_K_LOCAL,
            f"RP3betaSimpleBlend-category-l{lam}",
        )

        blend_rows.append(
            {
                "lambda": lam,
                "precision@10": metrics["precision_at_k"],
                "recall@10": metrics["recall_at_k"],
                "ndcg@10": metrics["ndcg_at_k"],
            }
        )

    simple_blend_df = pd.DataFrame(blend_rows).sort_values("ndcg@10", ascending=False).reset_index(drop=True)
    display(Markdown("## Validation sweep for the simple category-blend prototype"))
    display(simple_blend_df)
    print(
        "Interpretation:\n"
        "- This is the notebook-only Phase B prototype, not the final pushed repo path.\n"
        "- It tests the easiest defendable hybrid idea: base RP3beta score plus category-only metadata help.\n"
        "- Use its best row as the reference point before moving to the stronger pushed metadata implementations below."
    )
else:
    print("Set RUN_SIMPLE_BLEND = True if you want to run the notebook-only simple category-blend prototype.")

## Validation sweep for the simple category-blend prototype

,lambda,precision@10,recall@10,ndcg@10
0,0.9,0.040209,0.065338,0.081985
1,0.8,0.040235,0.064258,0.080920
2,0.7,0.037711,0.058499,0.073680
3,0.6,0.028475,0.040608,0.053073


Interpretation:
- This is the notebook-only Phase B prototype, not the final pushed repo path.
- It tests the easiest defendable hybrid idea: base RP3beta score plus category-only metadata help.
- Use its best row as the reference point before moving to the stronger pushed metadata implementations below.


## Variant Selection Notes

The base RP3beta model remains the clean interaction-only graph anchor. The metadata sections are included to show how the variant was extended after the shared foundation was already working.

Selection logic used in this notebook:

- Keep the base model as the reference point when measuring whether metadata actually helps.
- Treat the simple category blend as a transparent prototype, not the final production path.
- Prefer the pushed richer metadata implementation when it improves the shared metrics and remains explainable.
- Preserve all stages because the progression clarifies why the final metadata-aware version was chosen.

In [6]:
from analysis.shared_utils import RESULTS_DIR

rp3_meta_val = pd.read_csv(RESULTS_DIR / "rp3beta_metadata_val_summary.csv")
rp3_meta_test = pd.read_csv(RESULTS_DIR / "rp3beta_metadata_test_best.csv")
rp3_meta_best_val = pd.read_csv(RESULTS_DIR / "rp3beta_metadata_best_val.csv")
rp3_meta_best_test = pd.read_csv(RESULTS_DIR / "rp3beta_metadata_best_test.csv")

base_rp3_test = retained_test.loc[retained_test["model"] == "RP3beta-a0.9-b0.4-t400", ["model", "precision@10", "recall@10", "ndcg@10"]].copy()
base_rp3_test["story"] = "Base retained interaction-only RP3beta"

closest_simple_val = rp3_meta_val.loc[rp3_meta_val["metadata_mode"] == "category"].sort_values("ndcg@10", ascending=False).head(1).copy()
closest_simple_val["story"] = "Closest repo analog to the earlier simple category-only blend idea"

best_sweep_test = rp3_meta_test.copy()
best_sweep_test["story"] = "Best sweep winner from metadata-similarity fusion / reranking"

best_rich_test = rp3_meta_best_test.copy()
best_rich_test["story"] = "Best pushed richer metadata/content implementation"

metadata_story_table = pd.concat(
    [
        base_rp3_test,
        best_sweep_test[["model", "precision@10", "recall@10", "ndcg@10"]].assign(story=best_sweep_test["story"].iloc[0]),
        best_rich_test[["model", "precision@10", "recall@10", "ndcg@10"]].assign(story=best_rich_test["story"].iloc[0]),
    ],
    ignore_index=True,
)
base_ndcg = float(base_rp3_test["ndcg@10"].iloc[0])
metadata_story_table["ndcg_delta_vs_base_rp3beta"] = metadata_story_table["ndcg@10"] - base_ndcg

display(Markdown("## Metadata transition for RP3beta"))
display(closest_simple_val[["model", "metadata_mode", "fusion", "alpha", "precision@10", "recall@10", "ndcg@10", "story"]])

display(Markdown("## Test comparison: base RP3beta vs pushed metadata paths"))
display(metadata_story_table.sort_values("ndcg@10", ascending=False))

print(
    "Interpretation:\n"
    "- The simple category-only family is useful as the starting idea, but it is not the strongest pushed RP3beta metadata path.\n"
    "- The metadata sweep winner uses category_char + rerank with a very small metadata weight.\n"
    "- The richer pushed method, RP3betaMetaLowRank-categoryContent, is the strongest metadata-aware RP3beta result in this repo."
)

## Metadata transition for RP3beta

,model,metadata_mode,fusion,alpha,precision@10,recall@10,ndcg@10,story
2,RP3betaMeta-category-rerank-a0.1,category,rerank,0.1,0.039905,0.064404,0.081313,Closest repo analog to the earlier simple cate...


## Test comparison: base RP3beta vs pushed metadata paths

,model,precision@10,recall@10,ndcg@10,story,ndcg_delta_vs_base_rp3beta
2,RP3betaMetaLowRank-categoryContent,0.043602,0.066083,0.086520,Best pushed richer metadata/content implementa...,0.001488
0,RP3beta-a0.9-b0.4-t400,0.043121,0.064997,0.085032,Base retained interaction-only RP3beta,0.000000
1,RP3betaMeta-category_char-rerank-a0.1,0.043404,0.064650,0.084975,Best sweep winner from metadata-similarity fus...,-0.000057


Interpretation:
- The simple category-only family is useful as the starting idea, but it is not the strongest pushed RP3beta metadata path.
- The metadata sweep winner uses category_char + rerank with a very small metadata weight.
- The richer pushed method, RP3betaMetaLowRank-categoryContent, is the strongest metadata-aware RP3beta result in this repo.


## RP3beta Summary

The RP3beta work follows a clear implementation path:

1. **Base RP3beta** establishes a strong graph-only model using user-item interaction structure and a popularity penalty.
2. **Simple category blend** tests the most interpretable metadata extension by combining base RP3beta scores with category-only metadata scores.
3. **Metadata similarity reranking/fusion** uses richer metadata-derived item similarity graphs from category, package-word, and character n-gram features.
4. **Low-rank content + prior augmentation** is the strongest pushed RP3beta metadata implementation, combining graph scores with content profiles and a small metadata prior.

This notebook keeps each stage visible so a reader can see how the final metadata-aware RP3beta result evolved from the original graph-based model.